# Activity Cycle Diagram to ASM Translation

This notebook demonstrates the translation of Activity Cycle Diagram (ACD) models to Abstract State Machines (ASM) as implemented in SimASM.

**Reference:** Based on the formal translation algorithm in Section 7 of the paper.

## Overview

1. **Load ACD JSON** - Pure Activity Cycle Diagram specification (no SimASM syntax)
2. **Convert to SimASM** - Apply the translation algorithm (Algorithm 2)
3. **Run Little's Law Verification** - Empirically verify correctness
4. **Analyze Results** - Confirm $L = \lambda W$

## Key Concepts: Fine-Grained Activity Scanning

For trace equivalence with Event Graph models, we use **fine-grained semantics**:
- ONE atomic action per ASM step (start one activity OR execute one BTO event)
- Priority-based activity scanning for deterministic tie-breaking
- Separate timing phase and scanning phase

In [ ]:
# Install simasm (uncomment in Colab)
# !pip install simasm

import json
import simasm
from simasm.converter.acd import ACDSpec, load_acd_from_json, parse_pure_acd_json
from simasm.converter.acd import ACDConverter

print(f"SimASM version: {simasm.__version__}")

## 1. Activity Cycle Diagram Specification (JSON)

The ACD is specified in a pure JSON format using abstract operations.
This format contains **no SimASM syntax** - all code generation happens in the translator.

### M/M/5 Queue ACD Model

- **Queues (Passive States):** C (creator), Q (waiting), S (servers), Jobs (completed)
- **Activities (Active States):** Create (priority 1), Serve (priority 2)
- **Parameters:** $\lambda = 0.8$ (arrival rate), $\mu = 1.0$ (service rate), $N = 5$ (servers)

### ACD Formal Definition

An ACD is a tuple $\mathcal{N} = \langle A, Q, I, O, W^-, W^+, G, T, \mu_0 \rangle$ where:
- $A$ = set of activities (active states)
- $Q$ = set of queues (passive states)
- $I(a)$ = input queues for activity $a$
- $O(a)$ = output queues for activity $a$
- $W^-(a,q)$ = tokens consumed from queue $q$ when activity $a$ begins
- $W^+(a,q)$ = tokens produced to queue $q$ when activity $a$ ends
- $G(a)$ = guard condition for activity $a$
- $T(a)$ = duration of activity $a$
- $\mu_0$ = initial marking

In [ ]:
# M/M/5 Queue ACD Specification
mm5_acd_json = {
    "model_name": "mm5_acd",
    "description": "M/M/5 Queue using Activity Cycle Diagram formalism\nBased on Tocher (1960) and Carrie (1988)\nFine-grained step semantics for trace equivalence",

    "parameters": {
        "num_servers": {"type": "Nat", "value": 5, "description": "Number of servers (N in M/M/N)"},
        "iat_mean": {"type": "Real", "value": 1.25, "description": "Inter-arrival time mean (1/lambda)"},
        "ist_mean": {"type": "Real", "value": 1.0, "description": "Service time mean (1/mu)"},
        "sim_end_time": {"type": "Real", "value": 1000.0, "description": "Simulation end time"}
    },

    "token_types": {
        "Job": {
            "name": "Job",
            "parent": "Token",
            "attributes": {
                "arrival_time": {"type": "Real", "description": "Time job arrived"},
                "service_start_time": {"type": "Real", "description": "Time job started service"}
            },
            "description": "Customer job token"
        },
        "Resource": {
            "name": "Resource",
            "parent": "Token",
            "description": "Reusable resource token (creator, server)"
        }
    },

    "queues": {
        "C": {
            "initial_tokens": 1,
            "token_type": "Resource",
            "is_resource": True,
            "description": "Creator resource queue (bound activity driver)"
        },
        "Q": {
            "initial_tokens": 0,
            "token_type": "Job",
            "is_resource": False,
            "description": "Job waiting queue"
        },
        "S": {
            "initial_tokens": 5,
            "token_type": "Resource",
            "is_resource": True,
            "description": "Server resource queue"
        },
        "Jobs": {
            "initial_tokens": 0,
            "token_type": "Job",
            "is_resource": False,
            "description": "Completed jobs (sink)"
        }
    },

    "activities": [
        {
            "name": "Create",
            "priority": 1,
            "duration": "interarrival_time",
            "description": "Job creation activity (bound activity)",
            "at_begin": {
                "condition": "true",
                "consume": [
                    {"queue": "C", "count": 1, "bind_as": "creator_token"}
                ],
                "actions": []
            },
            "at_end": [
                {
                    "condition": "true",
                    "produce": [
                        {"queue": "C", "count": 1, "token_source": "creator_token"},
                        {"queue": "Q", "count": 1, "token_source": "new"}
                    ],
                    "actions": [
                        {"op": "increment", "var": "job_id_counter"},
                        {"op": "set_attribute", "entity": "new_token_0", "attribute": "id", "value": "job_id_counter"},
                        {"op": "set_attribute", "entity": "new_token_0", "attribute": "arrival_time", "value": "sim_clocktime"}
                    ]
                }
            ]
        },
        {
            "name": "Serve",
            "priority": 2,
            "duration": "service_time",
            "description": "Service activity (conditional activity)",
            "at_begin": {
                "condition": "true",
                "consume": [
                    {"queue": "S", "count": 1, "bind_as": "server_token"},
                    {"queue": "Q", "count": 1, "bind_as": "job_token"}
                ],
                "actions": [
                    {"op": "set_attribute", "entity": "job_token", "attribute": "service_start_time", "value": "sim_clocktime"}
                ]
            },
            "at_end": [
                {
                    "condition": "true",
                    "produce": [
                        {"queue": "S", "count": 1, "token_source": "server_token"},
                        {"queue": "Jobs", "count": 1, "token_source": "job_token"}
                    ],
                    "actions": [
                        {"op": "increment", "var": "departure_count"},
                        {"op": "compute", "var": "time_in_system", "expression": "sim_clocktime - arrival_time(job_token)"},
                        {"op": "compute", "var": "time_in_queue", "expression": "service_start_time(job_token) - arrival_time(job_token)"},
                        {"op": "accumulate", "var": "total_sojourn_time", "value": "time_in_system"},
                        {"op": "accumulate", "var": "total_time_in_queue", "value": "time_in_queue"}
                    ]
                }
            ]
        }
    ],

    "random_streams": {
        "interarrival_time": {
            "distribution": "exponential",
            "params": {"mean": "iat_mean"},
            "stream_name": "arrivals"
        },
        "service_time": {
            "distribution": "exponential",
            "params": {"mean": "ist_mean"},
            "stream_name": "service"
        }
    },

    "state_variables": {
        "job_id_counter": {"type": "Nat", "initial": 0},
        "departure_count": {"type": "Nat", "initial": 0},
        "total_sojourn_time": {"type": "Real", "initial": 0.0},
        "total_time_in_queue": {"type": "Real", "initial": 0.0}
    },

    "stopping_condition": "sim_clocktime >= sim_end_time",

    "observables": {
        "queue_count": {
            "name": "queue_count",
            "return_type": "Nat",
            "expression": "marking(Q)",
            "description": "Number in queue"
        },
        "servers_busy": {
            "name": "servers_busy",
            "return_type": "Nat",
            "expression": "num_servers - marking(S)",
            "description": "Number of busy servers"
        },
        "in_system": {
            "name": "in_system",
            "return_type": "Nat",
            "expression": "marking(Q) + (num_servers - marking(S))",
            "description": "Total in system"
        }
    },

    "fine_grained_semantics": {
        "description": "Fine-grained Activity Scanning for trace equivalence with Event Graph",
        "phase_sequence": ["init", "scan", "time", "execute"],
        "scanning_rule": "Try ONE activity per ASM step in priority order",
        "activity_order": "Activities scanned by priority (lower = first): Create(1) before Serve(2)",
        "separation": "Timing phase and scanning phase are SEPARATE ASM steps"
    }
}

print("ACD Model Components:")
print(f"  Queues (passive states): {list(mm5_acd_json['queues'].keys())}")
print(f"  Activities (active states): {[a['name'] for a in mm5_acd_json['activities']]}")
print(f"  Activity priorities: {[(a['name'], a['priority']) for a in mm5_acd_json['activities']]}")
print(f"  Token types: {list(mm5_acd_json['token_types'].keys())}")

## 2. Translation to SimASM

The `ACDConverter` class implements Algorithm 2 from the paper:

1. Construct signature $\Sigma$ (domains, functions)
2. Construct initial state $I_\mathcal{N}$
3. Construct initialization rule $P_{Init}$
4. For each activity $a \in A$, construct at-begin rule $P_{B,a}$
5. For each activity $a \in A$, construct at-end rule $P_{E,a}$
6. Construct at-begin conditions for each activity
7. Construct scanning rule $P_S$ (fine-grained: ONE activity per step)
8. Construct timing rule $P_T$
9. Construct executing rule $P_X$
10. Construct main rule $P_M$ (phase-based state machine)

### Phase-Based State Machine

```
init → scan → time → scan → time → ... → done
         ↺ (if activity started)
```

- **scan phase**: Try ONE activity in priority order; if started, stay in scan
- **time phase**: Pop next BTO event, advance clock, execute at-end actions

In [ ]:
# Parse JSON to ACDSpec
spec = parse_pure_acd_json(mm5_acd_json)

print("ACDSpec parsed successfully!")
print(f"  Model name: {spec.model_name}")
print(f"  Queues: {[q.name for q in spec.queues]}")
print(f"  Activities: {[a.name for a in spec.activities]}")
print(f"  Token types: {[t.name for t in spec.token_types]}")

In [ ]:
# Convert ACD to SimASM
converter = ACDConverter(spec)
simasm_code = converter.convert()

print(f"Generated {len(simasm_code.splitlines())} lines of SimASM code")
print("\n" + "="*70)
print("Generated SimASM Code (first 100 lines):")
print("="*70)
for i, line in enumerate(simasm_code.splitlines()[:100], 1):
    print(f"{i:4d} | {line}")
print("...")

In [ ]:
# Save generated code to file
output_path = "mm5_acd_translated.simasm"
with open(output_path, "w") as f:
    f.write(simasm_code)
print(f"Saved to: {output_path}")

## 3. Run Little's Law Experiment

Little's Law states: $L = \lambda W$

Where:
- $L$ = average number of jobs in the system
- $\lambda$ = arrival rate (1/1.25 = 0.8)
- $W$ = average time in the system

For M/M/5 with $\lambda = 0.8$, $\mu = 1.0$, $N = 5$:
- Traffic intensity: $a = \lambda/\mu = 0.8$
- Server utilization: $\rho = a/N = 0.16$

### Theoretical Values (M/M/N Queue)

For low utilization ($\rho = 0.16$), the probability of queueing is very small:
- $L \approx a = 0.8$ (most jobs go directly to service)
- $W \approx 1/\mu = 1.0$ (average service time)

In [ ]:
%%simasm experiment
// =============================================================================
// Little's Law Verification - Translated ACD M/M/5 Queue
// =============================================================================

experiment LittlesLawMM5ACD:
    model := "mm5_acd_translated.simasm"
    
    replication:
        count: 10
        warm_up_time: 100.0
        run_length: 1000.0
        seed_strategy: "incremental"
        base_seed: 12345
    endreplication
    
    statistics:
        // L: Average number in system (queue + in service)
        stat L_system: time_average
            expression: "marking(Q) + (num_servers - marking(S))"
        endstat
        
        // L_q: Average number in queue
        stat L_queue: time_average
            expression: "marking(Q)"
        endstat
        
        // L_s: Average number in service (busy servers)
        stat L_service: time_average
            expression: "num_servers - marking(S)"
        endstat

        // W: Average time in system
        stat W_system: count
            expression: "total_sojourn_time / departure_count"
        endstat

        // W_q: Average time in queue
        stat W_queue: count
            expression: "total_time_in_queue / departure_count"
        endstat
        
        // Total arrivals (for computing lambda)
        stat total_arrivals: count
            expression: "job_id_counter"
        endstat

        // Total departures
        stat total_departures: count
            expression: "departure_count"
        endstat
        
        // Server utilization (fraction of servers busy)
        stat rho_utilization: time_average
            expression: "(num_servers - marking(S)) / num_servers"
        endstat

        // Average idle servers
        stat idle_servers: time_average
            expression: "marking(S)"
        endstat
        
    endstatistics
    
    output:
        format: "json"
        file_path: "littles_law_mm5_acd_results.json"
    endoutput
endexperiment

## 4. Verify Little's Law

We verify that the translation is correct by checking Little's Law: $L = \lambda W$

In [ ]:
# Load results from the experiment output
import json

with open("littles_law_mm5_acd_results.json") as f:
    results = json.load(f)

# Theoretical values for M/M/5 (lambda=0.8, mu=1.0, N=5)
lambda_rate = 0.8  # arrival rate = 1/1.25
mu_rate = 1.0      # service rate = 1/1.0
N = 5              # number of servers
rho_theoretical = lambda_rate / (N * mu_rate)  # 0.16

# Get simulated values from summary statistics
summary = results.get("summary", {})
L_simulated = summary.get("L_system", {}).get("mean", 0)
W_simulated = summary.get("W_system", {}).get("mean", 0)
rho_simulated = summary.get("rho_utilization", {}).get("mean", 0)

# Calculate Little's Law verification
L_from_littles_law = lambda_rate * W_simulated

print("="*70)
print("Little's Law Verification: L = lambda * W")
print("="*70)
print(f"")
print(f"Parameters:")
print(f"  lambda (arrival rate)     = {lambda_rate}")
print(f"  mu (service rate)         = {mu_rate}")
print(f"  N (servers)               = {N}")
print(f"  rho (theoretical)         = {rho_theoretical:.4f}")
print(f"")
print(f"Simulated Values:")
print(f"  L (avg in system)         = {L_simulated:.4f}")
print(f"  W (avg time in system)    = {W_simulated:.4f}")
print(f"  rho (utilization)         = {rho_simulated:.4f}")
print(f"")
print(f"Little's Law Check:")
print(f"  L (simulated)             = {L_simulated:.4f}")
print(f"  lambda * W                = {L_from_littles_law:.4f}")
print(f"  Difference                = {abs(L_simulated - L_from_littles_law):.4f}")
if L_simulated > 0:
    error_pct = abs(L_simulated - L_from_littles_law) / L_simulated * 100
    print(f"  Relative Error            = {error_pct:.2f}%")
print(f"")

if L_simulated > 0 and abs(L_simulated - L_from_littles_law) / L_simulated < 0.05:
    print("VERIFICATION PASSED: Little's Law holds within 5% tolerance")
else:
    print("VERIFICATION WARNING: Check results (may need more replications)")

In [ ]:
# Compare with theoretical M/M/5 values
print("="*70)
print("Comparison with Theoretical M/M/5 Values")
print("="*70)
print(f"")
print(f"| Metric                   | Theoretical | Simulated | Error  |")
print(f"| ------------------------ | ----------- | --------- | ------ |")
print(f"| rho (utilization)        | {rho_theoretical:.3f}       | {rho_simulated:.3f}     | {abs(rho_simulated - rho_theoretical)/rho_theoretical*100:.2f}%  |")
print(f"| L (avg jobs in system)   | 0.800       | {L_simulated:.3f}     | {abs(L_simulated - 0.800)/0.800*100:.2f}%  |")
print(f"| W (avg time in system)   | 1.000       | {W_simulated:.3f}     | {abs(W_simulated - 1.000)/1.000*100:.2f}%  |")
if L_simulated > 0:
    littles_ratio = L_simulated / lambda_rate
    print(f"| Little's Law L/lambda    | 1.000       | {littles_ratio:.3f}     | {abs(littles_ratio - 1.000)/1.000*100:.2f}%  |")

## 5. Summary

This notebook demonstrated:

1. **Pure ACD JSON** - The Activity Cycle Diagram is specified using abstract operations without any SimASM syntax
2. **Automatic Translation** - The `ACDConverter` class implements Algorithm 2 to generate SimASM code
3. **Fine-Grained Semantics** - Phase-based state machine with one atomic action per ASM step
4. **Correctness Verification** - Little's Law ($L = \lambda W$) confirms the translation preserves the behavioral semantics

### ACD to ASM Correspondence (Table 6)

| ACD Element                     | ASM Construct                                              |
| ------------------------------- | ---------------------------------------------------------- |
| Queues $Q$                      | Domain `Queue`; constants for each queue                   |
| Activities $A$                  | Domain `Activity`; constants with priority attribute       |
| Marking $\mu$                   | Dynamic function: `marking(q: Queue): Nat`                 |
| Token lists                     | Dynamic function: `tokens(q: Queue): List<Token>`          |
| Input function $I(a)$           | At-begin consume specification                             |
| Output function $O(a)$          | At-end produce specification                               |
| Guard $G(a)$                    | At-begin condition in activity rule                        |
| Duration $T(a)$                 | BTO scheduled time: `sim_clocktime + T(a)`                 |
| BTO-events                      | Domain `BTOEvent`; properties: activity, time, bound tokens|
| Activity priority $\gamma(a)$   | Scanning order for fine-grained semantics                  |
| Phase 0 (Initialization)        | Initialization rule $P_{Init}$                             |
| Phase 1 (Scanning)              | Scanning rule $P_S$: try ONE activity per step             |
| Phase 2 (Timing)                | Timing rule $P_T$: sort FEL, pop BTO, advance clock        |
| Phase 3 (Executing)             | Executing rule $P_X$: process at-end actions               |

In [ ]:
# Clean up
%simasm_clear
print("Done.")